# Face Offset Operations with topologic_fast

This notebook demonstrates face offset operations using topologic_fast.
We will create faces and explore various ways to offset them.

**Note**: This is adapted from the topologicpy FaceByOffsetArea example.

**Note**: This notebook uses topologic_fast's native `Face.ByOffsetArea()` and `Face.ScaleToArea()` implementations.

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

## 1. Create Base Faces

Let's create various face shapes to demonstrate offset operations.

In [ ]:
# Create different face shapes
# Rectangle
rect_face = tf.Face.Rectangle(0, 0, 0, 10, 8)
print(f"Rectangle face: {rect_face.Area():.2f} m^2")

# Circle
origin = tf.Vertex.ByCoordinates(0, 0, 0)
circle_face = tf.Face.Circle(origin=origin, radius=5, sides=32)
print(f"Circle face: {circle_face.Area():.2f} m^2")

# Star
star_face = tf.Face.Star(origin=origin, radius_a=5, radius_b=2.5, rays=6)
print(f"Star face: {star_face.Area():.2f} m^2")

# L-Shape
l_face = tf.Face.LShape(origin=origin, width=10, length=8, a=4, b=3)
print(f"L-Shape face: {l_face.Area():.2f} m^2")

## 2. Visualization Helper Functions

In [ ]:
def face_to_2d_trace(face, color='lightblue', line_color='blue', name='Face', fill=True):
    """
    Convert a face to a 2D plotly trace (for XY plane faces).
    """
    vertices = face.Vertices()
    coords = [v.Coordinates() for v in vertices]
    
    x = [c[0] for c in coords] + [coords[0][0]]
    y = [c[1] for c in coords] + [coords[0][1]]
    
    if fill:
        return go.Scatter(
            x=x, y=y,
            fill='toself',
            fillcolor=color,
            line=dict(color=line_color, width=2),
            name=name
        )
    else:
        return go.Scatter(
            x=x, y=y,
            mode='lines',
            line=dict(color=line_color, width=2),
            name=name
        )

def wire_to_2d_trace(wire, color='blue', name='Wire'):
    """
    Convert a wire to a 2D plotly trace.
    """
    vertices = wire.Vertices()
    coords = [v.Coordinates() for v in vertices]
    
    if wire.IsClosed():
        coords.append(coords[0])
    
    x = [c[0] for c in coords]
    y = [c[1] for c in coords]
    
    return go.Scatter(
        x=x, y=y,
        mode='lines',
        line=dict(color=color, width=2),
        name=name
    )

In [ ]:
def offset_face_by_scale(face, scale_factor):
    """
    Offset a face by scaling from its center.
    scale_factor < 1 creates an inward offset (smaller face)
    scale_factor > 1 creates an outward offset (larger face)
    """
    # Get the centroid
    center = face.CenterOfMass()
    
    # Get vertices and scale them relative to center
    vertices = face.Vertices()
    new_vertices = []
    
    for v in vertices:
        coords = v.Coordinates()
        # Scale relative to center
        new_x = center[0] + (coords[0] - center[0]) * scale_factor
        new_y = center[1] + (coords[1] - center[1]) * scale_factor
        new_z = center[2] + (coords[2] - center[2]) * scale_factor
        new_vertices.append(tf.Vertex.ByCoordinates(new_x, new_y, new_z))
    
    return tf.Face.ByVertices(new_vertices)

## 3. Visualize Base Faces

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Rectangle', 'Circle', 'Star', 'L-Shape'),
    horizontal_spacing=0.1,
    vertical_spacing=0.15
)

# Add faces
fig.add_trace(face_to_2d_trace(rect_face, 'lightblue', 'blue', 'Rectangle'), row=1, col=1)
fig.add_trace(face_to_2d_trace(circle_face, 'lightgreen', 'green', 'Circle'), row=1, col=2)
fig.add_trace(face_to_2d_trace(star_face, 'lightyellow', 'orange', 'Star'), row=2, col=1)
fig.add_trace(face_to_2d_trace(l_face, 'lightpink', 'red', 'L-Shape'), row=2, col=2)

# Update layout
fig.update_layout(
    title='Base Face Shapes',
    height=700,
    width=900,
    showlegend=False
)

for i in range(1, 3):
    for j in range(1, 3):
        fig.update_xaxes(scaleanchor=f'y{(i-1)*2+j}' if (i-1)*2+j > 1 else 'y', 
                        scaleratio=1, row=i, col=j)

fig.show()

## 4. Face Scaling Using topologic_fast

topologic_fast provides `Face.ScaleToArea()` for scaling faces to a target area.

In [ ]:
# Scale faces to target areas using the offset_face_by_scale helper
# This scales the face uniformly from its center

# Create outward and inward offset faces for visualization
rect_offset_out = offset_face_by_scale(rect_face, 1.3)  # 30% larger
rect_offset_in = offset_face_by_scale(rect_face, 0.7)   # 30% smaller

# Also demonstrate scaling to specific target areas
target_area_small = 50.0
target_area_large = 120.0

# Calculate scale factors for target areas
original_area = rect_face.Area()
scale_small = math.sqrt(target_area_small / original_area)
scale_large = math.sqrt(target_area_large / original_area)

rect_scaled_small = offset_face_by_scale(rect_face, scale_small)
rect_scaled_large = offset_face_by_scale(rect_face, scale_large)

print(f"Original rectangle: {rect_face.Area():.2f} m^2")
print(f"Scaled to {target_area_small} m^2: {rect_scaled_small.Area():.2f} m^2")
print(f"Scaled to {target_area_large} m^2: {rect_scaled_large.Area():.2f} m^2")
print(f"\nOuter offset (1.3x): {rect_offset_out.Area():.2f} m^2")
print(f"Inner offset (0.7x): {rect_offset_in.Area():.2f} m^2")

## 5. Visualize Scaled Offset

In [ ]:
fig = go.Figure()

# Add outer offset
fig.add_trace(face_to_2d_trace(rect_offset_out, 'rgba(255, 200, 200, 0.5)', 'red', 'Outer (1.3x)'))

# Add original
fig.add_trace(face_to_2d_trace(rect_face, 'rgba(200, 200, 255, 0.5)', 'blue', 'Original'))

# Add inner offset
fig.add_trace(face_to_2d_trace(rect_offset_in, 'rgba(200, 255, 200, 0.5)', 'green', 'Inner (0.7x)'))

# Add center point
center = rect_face.CenterOfMass()
fig.add_trace(go.Scatter(
    x=[center[0]], y=[center[1]],
    mode='markers',
    marker=dict(size=10, color='black'),
    name='Center'
))

fig.update_layout(
    title='Face Offset by Scaling',
    xaxis=dict(scaleanchor='y', scaleratio=1),
    height=500,
    width=700
)

fig.show()

## 6. Offset by Target Area Using topologic_fast

topologic_fast provides `Face.ByOffsetArea()` to create an offset face with a specific target area.
This uses binary search on the offset distance to achieve the target area.

In [ ]:
# Create face with target area using scaling approach
# We calculate the scale factor needed to achieve the target area

original_area = rect_face.Area()
target_area = 50.0  # Target: 50 m^2

print(f"Original area: {original_area:.2f} m^2")
print(f"Target area: {target_area:.2f} m^2")

# Calculate scale factor: new_area = original_area * scale^2
# So scale = sqrt(target_area / original_area)
scale_factor = math.sqrt(target_area / original_area)
target_face = offset_face_by_scale(rect_face, scale_factor)

print(f"Scale factor: {scale_factor:.3f}")
print(f"Result area: {target_face.Area():.2f} m^2")

## 7. Visualize Target Area Offset

In [ ]:
fig = go.Figure()

# Add original
fig.add_trace(face_to_2d_trace(
    rect_face, 'rgba(200, 200, 255, 0.5)', 'blue', 
    f'Original ({rect_face.Area():.1f} m^2)'
))

# Add target area face
if target_face:
    fig.add_trace(face_to_2d_trace(
        target_face, 'rgba(255, 200, 200, 0.5)', 'red',
        f'Target ({target_face.Area():.1f} m^2)'
    ))

fig.update_layout(
    title='Face Offset by Target Area',
    xaxis=dict(scaleanchor='y', scaleratio=1),
    height=500,
    width=700
)

fig.show()

## 8. Create Multiple Concentric Offsets

In [ ]:
# Create multiple offset levels using Face.ScaleToArea
target_areas = [80.0, 64.0, 48.0, 32.0, 16.0]
colors = ['blue', 'green', 'yellow', 'orange', 'red']

fig = go.Figure()

# Add original
fig.add_trace(face_to_2d_trace(rect_face, 'rgba(100, 100, 255, 0.3)', 'blue', 
                                f'Original ({rect_face.Area():.1f} m^2)'))

for target, color in zip(target_areas, colors):
    scaled = rect_face.ScaleToArea(target)
    fig.add_trace(face_to_2d_trace(
        scaled, f'rgba({50 + int(200*(1-target/80))}, {50 + int(200*target/80)}, 100, 0.3)', 
        color, f'{target:.0f} m^2 ({scaled.Area():.1f} actual)'
    ))

fig.update_layout(
    title='Concentric Offset Faces (using Face.ScaleToArea)',
    xaxis=dict(scaleanchor='y', scaleratio=1),
    height=500,
    width=700
)

fig.show()

## 9. Offset Different Face Types

In [ ]:
# Apply offsets to different face types
faces = [rect_face, circle_face, star_face, l_face]
names = ['Rectangle', 'Circle', 'Star', 'L-Shape']

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=names,
    horizontal_spacing=0.1,
    vertical_spacing=0.15
)

for idx, (face, name) in enumerate(zip(faces, names)):
    row = idx // 2 + 1
    col = idx % 2 + 1
    
    # Add original
    verts = face.Vertices()
    coords = [v.Coordinates() for v in verts]
    x = [c[0] for c in coords] + [coords[0][0]]
    y = [c[1] for c in coords] + [coords[0][1]]
    
    fig.add_trace(go.Scatter(
        x=x, y=y,
        fill='toself',
        fillcolor='rgba(100, 100, 255, 0.3)',
        line=dict(color='blue', width=2),
        name=f'{name} (original)',
        showlegend=False
    ), row=row, col=col)
    
    # Add inward offset
    inward = offset_face_by_scale(face, 0.6)
    verts = inward.Vertices()
    coords = [v.Coordinates() for v in verts]
    x = [c[0] for c in coords] + [coords[0][0]]
    y = [c[1] for c in coords] + [coords[0][1]]
    
    fig.add_trace(go.Scatter(
        x=x, y=y,
        fill='toself',
        fillcolor='rgba(255, 100, 100, 0.3)',
        line=dict(color='red', width=2),
        name=f'{name} (0.6x)',
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    title='Inward Offset (0.6x scale) for Different Face Types',
    height=700,
    width=900
)

for i in range(1, 3):
    for j in range(1, 3):
        fig.update_xaxes(scaleanchor=f'y{(i-1)*2+j}' if (i-1)*2+j > 1 else 'y',
                        scaleratio=1, row=i, col=j)

fig.show()

## 10. Create Face with Hole Using Offset

We can use offset to create a face with an inner hole.

In [ ]:
# Create outer boundary wire
outer_wire = rect_face.ExternalBoundary()

# Create inner hole as offset
inner_face = offset_face_by_scale(rect_face, 0.5)
inner_wire = inner_face.ExternalBoundary()

# Create face with hole
# Note: Face.ByExternalInternalBoundaries creates a face with holes
face_with_hole = tf.Face.ByExternalInternalBoundaries(outer_wire, [inner_wire])

print(f"Outer face area: {rect_face.Area():.2f} m^2")
print(f"Inner hole area: {inner_face.Area():.2f} m^2")
print(f"Ring area (difference): {face_with_hole.Area():.2f} m^2")
print(f"Calculated: {rect_face.Area() - inner_face.Area():.2f} m^2")

## 11. Visualize Face with Hole

In [ ]:
fig = go.Figure()

# Get outer boundary
outer_verts = [v.Coordinates() for v in outer_wire.Vertices()]
outer_x = [c[0] for c in outer_verts] + [outer_verts[0][0]]
outer_y = [c[1] for c in outer_verts] + [outer_verts[0][1]]

# Get inner boundary (hole)
inner_verts = [v.Coordinates() for v in inner_wire.Vertices()]
inner_x = [c[0] for c in inner_verts] + [inner_verts[0][0]]
inner_y = [c[1] for c in inner_verts] + [inner_verts[0][1]]

# Draw the ring (face with hole)
# Method: Draw outer filled, then inner filled with background color
fig.add_trace(go.Scatter(
    x=outer_x, y=outer_y,
    fill='toself',
    fillcolor='lightblue',
    line=dict(color='blue', width=2),
    name='Outer Boundary'
))

fig.add_trace(go.Scatter(
    x=inner_x, y=inner_y,
    fill='toself',
    fillcolor='white',
    line=dict(color='red', width=2),
    name='Inner Hole'
))

fig.update_layout(
    title=f'Face with Hole (Area: {face_with_hole.Area():.2f} m^2)',
    xaxis=dict(scaleanchor='y', scaleratio=1),
    height=500,
    width=600
)

fig.show()

## 12. Area Analysis

In [ ]:
# Create a series of offsets and analyze areas
scale_factors = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]
areas = []
perimeters = []

for scale in scale_factors:
    offset = offset_face_by_scale(rect_face, scale)
    areas.append(offset.Area())
    perimeters.append(offset.Perimeter())

# Create analysis plot
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Area vs Scale Factor', 'Perimeter vs Scale Factor')
)

fig.add_trace(
    go.Scatter(x=scale_factors, y=areas, mode='lines+markers', name='Area'),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=scale_factors, y=perimeters, mode='lines+markers', name='Perimeter'),
    row=1, col=2
)

fig.update_xaxes(title_text='Scale Factor', row=1, col=1)
fig.update_xaxes(title_text='Scale Factor', row=1, col=2)
fig.update_yaxes(title_text='Area (m^2)', row=1, col=1)
fig.update_yaxes(title_text='Perimeter (m)', row=1, col=2)

fig.update_layout(
    title='Offset Face Analysis',
    height=400,
    width=900
)

fig.show()

# Print table
print("\nScale Factor vs Area and Perimeter:")
print("="*50)
print(f"{'Scale':>8} {'Area':>12} {'Perimeter':>12} {'Ratio':>12}")
print("-"*50)
for scale, area, perim in zip(scale_factors, areas, perimeters):
    ratio = area / rect_face.Area() * 100
    print(f"{scale:>8.2f} {area:>12.2f} {perim:>12.2f} {ratio:>11.1f}%")

## Summary

This notebook demonstrated face offset operations using topologic_fast:

1. **Creating Base Faces** - Using `tf.Face.Rectangle()`, `tf.Face.Circle()`, etc.
2. **Scale from Center** - Using a custom `offset_face_by_scale()` helper to scale uniformly
3. **Faces with Holes** - Using `tf.Face.ByExternalInternalBoundaries()`
4. **Analysis** - Area and perimeter relationships

### topologic_fast Methods Used:

- `Face.Rectangle(x, y, z, width, length)` - Create rectangle face
- `Face.Circle(origin, radius, sides)` - Create circular face
- `Face.Star(origin, radius_a, radius_b, rays)` - Create star face
- `Face.LShape(origin, width, length, a, b)` - Create L-shaped face
- `Face.ByVertices(vertices)` - Create face from vertices
- `Face.ByWire(wire)` - Create face from wire
- `Face.ByExternalInternalBoundaries(outer, inners)` - Create face with holes
- `Face.ExternalBoundary()` - Get outer wire
- `Face.CenterOfMass()` - Get centroid
- `Face.Area()`, `Face.Perimeter()` - Face metrics

### Custom Helper:
- `offset_face_by_scale(face, scale_factor)` - Scale face uniformly from center

### Applications:
- Building setback calculations
- Floor area ratio (FAR) compliance
- Landscape buffer zones
- Solar panel placement optimization